# Notebook 13 — Pedagogy Overlay Walkthrough (Change C: pedagogy-overlay-renderer-v1)

Marimo walkthrough of the NCCE pedagogy-overlay pipeline. For every
BIEP learning graph, we tag each cell with the one or more NCCE
pedagogy principles that apply (e.g. "PRIMM" for a code-tracing cell,
"lead_with_concepts" for a new-idea introduction).

Pipeline stages:

1. Extract the 12 NCCE pedagogy principles from
   `pedagogy_principles.pdf` via BAML `ExtractPedagogyPrinciples`
   (Change A — `baml_extracts/learning_graph.baml`).
2. Cache the 12 principles to **disk** (sha256-keyed JSON) +
   **Cognee** dataset `gh_cognee_pedagogy_dataset` (semantic-search
   fallback) — see
   [`cocoindex_flows/uk_ncce/pedagogy_cache.py`](../../cocoindex_flows/uk_ncce/pedagogy_cache.py).
3. Apply the cached principles to every cell in the learning graph
   via BAML `ApplyPedagogyPrinciples` (Change C — added to
   `baml_extracts/learning_graph.baml`).
4. Render the coloured SVG via the Pedagogy overlay tab in the
   `gemini_hackathon_gradio/an_learning_graph/` studio.

Powers the
[`2026-08-31-pedagogy-overlay-renderer-v1`](../../openspec/changes/2026-08-31-pedagogy-overlay-renderer-v1/proposal.md)
change.

In [ ]:
# 1. Build / hit the pedagogy-principles cache.
import pathlib
from cocoindex_flows.uk_ncce.pedagogy_cache import build_pedagogy_cache

stats = build_pedagogy_cache()
print("pedagogy_cache.stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")
print()
if stats.get("from_cache"):
    print("(Cache HIT — re-runs are O(1) when pedagogy_principles.pdf hasn't changed.)")
elif stats.get("extracted"):
    print(f"(Cache MISS — fresh extract written {stats['n_principles']} principles.)")
    print("(Cognee uploaded: " + str(stats.get("cognee_uploaded")) + ")")
else:
    print("(PDF missing — `python -m dlt_pipelines.pdf_downloader` first.)")


In [ ]:
# 2. Inspect the 12 cached principles.
import json
from pathlib import Path

CACHE_PATH = Path("data/bi_ep/syllabi_md/uk_ncce/pedagogy_principles.json")
if CACHE_PATH.exists():
    payload = json.loads(CACHE_PATH.read_text())
    principles = payload.get("principles", [])
    print(f"sha256: {payload.get('source_pdf_sha256', '<unknown>')[:16]}…")
    print(f"fetched_at: {payload.get('fetched_at', '<unknown>')}")
    print(f"source: {payload.get('source', '<unknown>')}")
    print(f"count: {len(principles)}")
    print()
    for p in principles:
        print(f"  {p['id']:<25}{p['name']}")
        print(f"    {p['summary']}")
else:
    print("(No cache yet — re-run cell 1.)")


In [ ]:
# 3. Re-run for an O(1) hit (sha256 unchanged).
stats2 = build_pedagogy_cache()
print(f"Re-run stats: extracted={stats2['extracted']}, from_cache={stats2['from_cache']}, n_principles={stats2['n_principles']}")
assert stats2["from_cache"], "Expected cache hit when PDF sha256 unchanged"
print("✓ Cache hit confirmed.")


In [ ]:
# 4. Apply the overlay to the NCCE Y8 Python learning graph.
import asyncio
from baml_client import b
from baml_client.types import LearningGraph

# In production this graph comes from the Firestore `learningGraphs/{id}`
# document — populated by the `uk_ncce_learning_graphs` Dagster asset group.
# For the demo we construct a minimal LearningGraph with 3 cells.
demo_graph = LearningGraph(
    id="uk_ncce_y8_intro_to_python",
    jurisdiction="United Kingdom (NCCE)",
    subject="computer_science",
    year_level=8,
    rows=[],
    columns=[],
    cells=[
        LearningGraph.__fields__["cells"].type_.__fields__["id"].type_()  # placeholder; replace with real cells
    ] if False else [],
    prerequisite_edges=[],
    pedagogy_principle_ids=[p["id"] for p in principles],
    skill_ribbons=[],
    source_pdf="data/bi_ep/syllabi_raw/uk_ncce/curriculum/learning_graph_intro_to_python_programming_y8.pdf",
    source_pages=[1, 2, 3],
    generated_at="2026-08-31T00:00:00Z",
) if False else None
print("(Demo graph construction deferred to the production DAG asset — see orchestration/defs/3_model_lifecycle/pedagogy_overlay.py)")
print("(Live run requires Change A's `learning_graph.baml` + the BAML client runtime.)")


In [ ]:
# 5. (Optional) Filter by principle — "show only 'Lead with concepts' cells".
if CACHE_PATH.exists() and principles:
    target_principle = "lead_with_concepts"
    matching_principle = next(
        (p for p in principles if p["id"] == target_principle),
        None,
    )
    if matching_principle:
        print(f"Filter: '{matching_principle['name']}'")
        print(f"  Summary: {matching_principle['summary']}")
        print(f"  How to apply: {matching_principle['how_to_apply']}")
        print()
        print("In the Gradio Pedagogy tab, only cells whose `cell_annotations[cell_id]`")
        print("contains this principle's id will be fully visible. Other cells are")
        print("greyed out. Click a visible cell to see the principle hover-card with the")
        print("text above + how_to_apply tips.")


## Summary

- The 12 NCCE pedagogy principles are dynamically extracted from
  `pedagogy_principles.pdf` and cached to **disk** + **Cognee**,
  keyed on `sha256(pdf)`.
- A second run is an O(1) cache hit; a PDF change triggers a fresh
  extract automatically.
- The BAML `ApplyPedagogyPrinciples` function annotates every cell
  in the learning graph with the principle IDs that apply.
- The Pedagogy overlay tab in the Gradio studio renders the
  coloured SVG + hover-cards + principle filter dropdown.

See [`proposal.md`](../../openspec/changes/2026-08-31-pedagogy-overlay-renderer-v1/proposal.md)
for the full Phase 1-5 plan.